# 📘 Module 5.2 – Motion Detection

📌 Goal: Detect movement in real-time video

### CORE IDEA (VERY IMPORTANT)

Motion = change between frames

- OpenCV detects motion by:
- Comparing frames
- Separating foreground from background
- Removing noise

🔹 MOTION DETECTION TECHNIQUES YOU WILL LEARN

- ✔ Frame differencing
- ✔ Background subtraction
- ✔ Noise handling using morphology 

## ✅ COMPLETE SINGLE-FILE CODE

In [1]:
import cv2
import numpy as np

# ================================
# STEP 1: OPEN WEBCAM
# ================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Error: Webcam not accessible")
    exit()

# ================================
# STEP 2: BACKGROUND SUBTRACTOR
# ================================
# MOG2 is adaptive and robust

bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=16,
    detectShadows=True
)

# ================================
# STEP 3: READ FIRST FRAME
# (For frame differencing)
# ================================

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

# ================================
# STRUCTURING ELEMENT FOR NOISE REMOVAL
# ================================

kernel = np.ones((5, 5), np.uint8)

# ================================
# STEP 4: MAIN LOOP
# ================================

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # Resize for performance
    frame = cv2.resize(frame, (640, 480))

    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ==================================================
    # PART 1: FRAME DIFFERENCING
    # ==================================================

    # Absolute difference between current & previous frame
    diff = cv2.absdiff(prev_gray, gray)

    # Threshold the difference image
    _, diff_thresh = cv2.threshold(
        diff, 30, 255, cv2.THRESH_BINARY
    )

    # Noise removal using morphology
    diff_thresh = cv2.morphologyEx(
        diff_thresh, cv2.MORPH_OPEN, kernel
    )

    # ==================================================
    # PART 2: BACKGROUND SUBTRACTION
    # ==================================================

    fg_mask = bg_subtractor.apply(frame)

    # Remove noise
    fg_mask = cv2.morphologyEx(
        fg_mask, cv2.MORPH_OPEN, kernel
    )

    # ==================================================
    # DISPLAY RESULTS
    # ==================================================

    cv2.imshow("Original Frame", frame)
    cv2.imshow("Frame Differencing", diff_thresh)
    cv2.imshow("Background Subtraction", fg_mask)

    # Update previous frame
    prev_gray = gray.copy()

    # Press 'q' to quit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ================================
# CLEANUP
# ================================

cap.release()
cv2.destroyAllWindows()


### EXPLANATION (SIMPLE & EXAM-READY)
#### 🔹 Frame Differencing
diff = cv2.absdiff(prev_gray, gray)


- ✔ Detects sudden changes
- ✔ Simple but sensitive to noise

#### 🔹 Background Subtraction
bg_subtractor = cv2.createBackgroundSubtractorMOG2()


- ✔ Learns background over time
- ✔ Best for real-time surveillance

#### 🔹 Noise Handling
cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)


- ✔ Removes small false motion
- ✔ Essential in real systems

##  REAL-WORLD APPLICATIONS

- ✔ CCTV surveillance
- ✔ Motion-based alarms
- ✔ Gesture detection
- ✔ Smart cameras
- ✔ Traffic monitoring

# Mandatory Practice Included

- ✔ Edge detection on webcam
- ✔ FPS display
- ✔ Resize before display
- ✔ Motion detection with bounding boxes
- ✔ Large-area filtering

In [2]:
import cv2
import numpy as np
import time

# ==============================
# VIDEO SOURCE (WEBCAM)
# ==============================
cap = cv2.VideoCapture(0)

# ==============================
# VIDEO WRITER (Grayscale Recording)
# ==============================
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(
    'grayscale_output.avi',
    fourcc,
    20.0,
    (640, 480),
    isColor=False
)

# ==============================
# BACKGROUND SUBTRACTOR
# ==============================
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

# ==============================
# FPS CALCULATION
# ==============================
prev_time = 0

print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ==============================
    # RESIZE FRAME (Mandatory)
    # ==============================
    frame = cv2.resize(frame, (640, 480))

    # ==============================
    # GRAYSCALE CONVERSION
    # ==============================
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ==============================
    # RECORD GRAYSCALE VIDEO
    # ==============================
    out.write(gray)

    # ==============================
    # EDGE DETECTION (Mandatory)
    # ==============================
    edges = cv2.Canny(gray, 100, 200)

    # ==============================
    # MOTION DETECTION
    # ==============================
    fg_mask = bg_subtractor.apply(frame)

    # Threshold value changed (30 → 50)
    _, thresh = cv2.threshold(fg_mask, 50, 255, cv2.THRESH_BINARY)

    # Increase kernel size (Noise removal)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    # ==============================
    # FIND CONTOURS (Motion Areas)
    # ==============================
    contours, _ = cv2.findContours(
        cleaned,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Detect motion only if LARGE area moves
        if area > 1500:
            x, y, w, h = cv2.boundingRect(cnt)

            # Draw bounding box on motion
            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 0, 255),
                2
            )

            cv2.putText(
                frame,
                "MOTION",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

    # ==============================
    # FPS DISPLAY (Mandatory)
    # ==============================
    current_time = time.time()
    fps = 1 / (current_time - prev_time) if prev_time else 0
    prev_time = current_time

    cv2.putText(
        frame,
        f"FPS: {int(fps)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (0, 255, 0),
        2
    )

    # ==============================
    # DISPLAY WINDOWS
    # ==============================
    cv2.imshow("Original Frame", frame)
    cv2.imshow("Edges", edges)
    cv2.imshow("Motion Mask", cleaned)

    # ==============================
    # EXIT
    # ==============================
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==============================
# RELEASE RESOURCES
# ==============================
cap.release()
out.release()
cv2.destroyAllWindows()


Press 'q' to quit


### 🎯 WHAT YOU MASTERED IN PHASE 5

- ✔ Real-time video processing
- ✔ Webcam + file handling
- ✔ FPS calculation
- ✔ Edge detection on live feed
- ✔ Motion detection with noise filtering
- ✔ Bounding boxes on moving objects
- ✔ Grayscale video recording
- ✔ Static & dynamic background handling

# MINI PROJECT 2 – OBJECT TRACKING CAMERA
🎯 Objective

Track moving objects smoothly using Optical Flow, simulating a basic tracking camera.

🔍 Concepts Used

- Lucas–Kanade Optical Flow
- Feature point detection
- Motion vector visualization
- Real-time tracking

In [3]:
import cv2
import numpy as np
import time

# ==============================
# VIDEO SOURCE
# ==============================
cap = cv2.VideoCapture(0)

# ==============================
# VIDEO WRITER (Grayscale)
# ==============================
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out = cv2.VideoWriter(
    'grayscale_output.avi',
    fourcc,
    20.0,
    (640, 480),
    isColor=False
)

# ==============================
# BACKGROUND SUBTRACTOR
# ==============================
bg_subtractor = cv2.createBackgroundSubtractorMOG2(
    history=500,
    varThreshold=50,
    detectShadows=True
)

# ==============================
# OPTICAL FLOW PARAMETERS
# ==============================
feature_params = dict(
    maxCorners=100,
    qualityLevel=0.3,
    minDistance=7,
    blockSize=7
)

lk_params = dict(
    winSize=(15, 15),
    maxLevel=2,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03)
)

# ==============================
# INITIAL FRAME FOR TRACKING
# ==============================
ret, old_frame = cap.read()
old_frame = cv2.resize(old_frame, (640, 480))
old_gray = cv2.cvtColor(old_frame, cv2.COLOR_BGR2GRAY)
p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

mask = np.zeros_like(old_frame)

# ==============================
# FPS
# ==============================
prev_time = 0

print("Press 'q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (640, 480))
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # ==============================
    # RECORD GRAYSCALE VIDEO
    # ==============================
    out.write(gray)

    # ==============================
    # EDGE DETECTION
    # ==============================
    edges = cv2.Canny(gray, 100, 200)

    # =====================================================
    # MINI PROJECT 1 – MOTION DETECTION SYSTEM
    # =====================================================
    fg_mask = bg_subtractor.apply(frame)

    _, thresh = cv2.threshold(fg_mask, 50, 255, cv2.THRESH_BINARY)

    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (7, 7))
    cleaned = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel)

    contours, _ = cv2.findContours(
        cleaned,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )

    for cnt in contours:
        area = cv2.contourArea(cnt)

        # Detect motion only if large area moves
        if area > 1500:
            x, y, w, h = cv2.boundingRect(cnt)

            cv2.rectangle(
                frame,
                (x, y),
                (x + w, y + h),
                (0, 0, 255),
                2
            )

            cv2.putText(
                frame,
                "MOTION",
                (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (0, 0, 255),
                2
            )

    # =====================================================
    # MINI PROJECT 2 – OBJECT TRACKING CAMERA
    # =====================================================
    if p0 is not None:
        p1, st, err = cv2.calcOpticalFlowPyrLK(
            old_gray,
            gray,
            p0,
            None,
            **lk_params
        )

        good_new = p1[st == 1]
        good_old = p0[st == 1]

        for new, old in zip(good_new, good_old):
            a, b = new.ravel()
            c, d = old.ravel()

            mask = cv2.line(
                mask,
                (int(a), int(b)),
                (int(c), int(d)),
                (0, 255, 0),
                2
            )
            frame = cv2.circle(
                frame,
                (int(a), int(b)),
                5,
                (0, 0, 255),
                -1
            )

        tracking_frame = cv2.add(frame, mask)
    else:
        tracking_frame = frame

    old_gray = gray.copy()
    p0 = cv2.goodFeaturesToTrack(old_gray, mask=None, **feature_params)

    # ==============================
    # FPS DISPLAY
    # ==============================
    current_time = time.time()
    fps = 1 / (current_time - prev_time) if prev_time else 0
    prev_time = current_time

    cv2.putText(
        tracking_frame,
        f"FPS: {int(fps)}",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.8,
        (255, 255, 0),
        2
    )

    # ==============================
    # DISPLAY WINDOWS
    # ==============================
    cv2.imshow("Motion Detection System", frame)
    cv2.imshow("Object Tracking Camera", tracking_frame)
    cv2.imshow("Edges", edges)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# ==============================
# CLEANUP
# ==============================
cap.release()
out.release()
cv2.destroyAllWindows()


Press 'q' to quit
